# Notebook 3 — AI-Assisted Research & Development with DeepSeek

**Workshop:** Edge Intelligence, Bio-Inspired Optimization & UAV-Assisted VANETs — ITS IRM 2026
**Estimated time:** 60–75 minutes, fully self-paced
**Prerequisite:** Notebook 2 (Exercise C reuses your GWO clustering code)

### What you'll do
Three exercises, each pairing a DeepSeek prompt with an **automated verification step** — the goal is
fluency with LLM-assisted research, not blind trust in it.

| Exercise | Skill | Verified by |
|---|---|---|
| A — Debugging | Using an LLM to fix real code | Automated test cases (assert-based) |
| B — Literature synthesis | Using an LLM for lit review without hallucinating citations | An automated quote-verification checker you'll use on DeepSeek's actual output |
| C — Algorithm design | Using an LLM to brainstorm, then implementing and benchmarking the idea yourself | Your own benchmark from Notebook 2 |

### Getting a DeepSeek API key (2 minutes, one-time)
1. Go to **platform.deepseek.com** → sign up or log in
2. Navigate to **API Keys** → create a new key
3. Run the setup cell below and paste your key when prompted (it is never printed or saved to this file)

If your venue's network blocks API access, use **chat.deepseek.com** in a browser tab instead — copy each
prompt below into the chat, then paste DeepSeek's response into the indicated cell.

### Setup — run this first

In [ ]:
!pip install -q openai

import re
from getpass import getpass

try:
    from openai import OpenAI
    _api_key = getpass("Paste your DeepSeek API key (input hidden), then press Enter: ")
    client = OpenAI(api_key=_api_key, base_url="https://api.deepseek.com")

    def ask_deepseek(prompt, system=None, model="deepseek-v4-flash"):
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        resp = client.chat.completions.create(model=model, messages=messages)
        return resp.choices[0].message.content

    print("DeepSeek client ready.")
except Exception as e:
    print("Could not set up the API client:", e)
    print("No problem — use chat.deepseek.com in a browser tab instead, and paste responses")
    print("into the 'PASTE_RESPONSE_HERE' cells below.")

## Exercise A — DeepSeek-Assisted Debugging (20 min)

The function below computes **cluster-head stability** — the fraction of rounds where the same
cluster-head set persisted from the previous round. It has a real bug.

**Step 1.** Run the buggy function and see it produce a wrong answer.

In [ ]:
def compute_cluster_stability_BUGGY(head_history):
    """head_history: list of sets, one per round, containing cluster-head node ids for that round.
    Returns the fraction of rounds where the head set matched the previous round."""
    same_count = 0
    for i in range(len(head_history)):
        if head_history[i] == head_history[i - 1]:
            same_count += 1
    return same_count / len(head_history)

test_history = [{1, 2, 3}, {1, 2, 3}, {4, 5, 6}, {4, 5, 6}, {4, 5, 6}]
print("Buggy result:", compute_cluster_stability_BUGGY(test_history))
print("(By hand: rounds 1-4 compared to the previous round give 3 matches out of 4 comparisons = 0.75)")

**Step 2.** Ask DeepSeek to find and fix it. Run the next cell (or copy the printed prompt into
chat.deepseek.com), then paste the fixed function into the cell after that.

In [ ]:
import inspect

debug_prompt = f'''This Python function should return the fraction of rounds where the cluster-head
set matches the PREVIOUS round, comparing round 1 against round 0, round 2 against round 1, and so on
(the very first round has no previous round to compare against, so it should be excluded). It gives an
incorrect result. Find the bug, fix it, and explain what was wrong in plain language.

{inspect.getsource(compute_cluster_stability_BUGGY)}'''
print(debug_prompt)

try:
    print("\n--- DeepSeek's response ---\n")
    print(ask_deepseek(debug_prompt))
except NameError:
    print("(ask_deepseek not available — paste the prompt above into chat.deepseek.com instead)")

In [ ]:
# PASTE_RESPONSE_HERE: replace the body of this function with DeepSeek's fixed version
def compute_cluster_stability_FIXED(head_history):
    # YOUR FIX (from DeepSeek) HERE
    pass

### Self-check — run this to verify the fix

In [ ]:
_passed = True
_result = compute_cluster_stability_FIXED(test_history)
if _result is None:
    print("compute_cluster_stability_FIXED returned None — did DeepSeek's fix get pasted in correctly?")
    _passed = False
elif abs(_result - 0.75) > 1e-9:
    print(f"Expected 0.75, got {_result}. DeepSeek's fix may not be fully correct — try asking it")
    print("to double check the loop's start index and the denominator.")
    _passed = False

# a second, independent test case to guard against a fix that only works by coincidence
test_history_2 = [{1}, {2}, {2}, {2}, {1}, {1}]
_expected_2 = 0.6  # matches at i=2 (2==2), i=3 (2==2)... let's verify: rounds compared: (2,1)F (2,2)T (2,2)T? wait recompute
_result_2 = compute_cluster_stability_FIXED(test_history_2)
# comparisons: r1 vs r0: {2}=={1}? F | r2 vs r1: {2}=={2}? T | r3 vs r2: {2}=={2}? T | r4 vs r3: {1}=={2}? F | r5 vs r4: {1}=={1}? T
# matches = 3 out of 5 comparisons = 0.6
if _result_2 is not None and abs(_result_2 - 0.6) > 1e-9:
    print(f"Second test case failed: expected 0.6, got {_result_2}.")
    _passed = False

if _passed:
    print("Exercise A self-check: ALL TESTS PASSED ✅")
    print("DeepSeek's fix (as you applied it) is verified correct against two independent test cases.")
assert _passed

**Discussion (no code needed):** Did DeepSeek's explanation correctly diagnose *why* the original
code was wrong (the off-by-one wraparound and the denominator), or did it just patch symptoms? What
would you have had to check even if the fix had looked plausible?

## Exercise B — Literature Synthesis Without Hallucinating (15 min)

You'll ask DeepSeek to connect three abstracts, then **automatically check every quote it gives you**
against the real source text — this is the core academic-integrity skill for using LLMs in a lit review.

In [ ]:
abstracts = {
    "J4 (CAMONET)": (
        "Vehicular ad-hoc networks require efficient clustering to reduce control overhead in highly "
        "dynamic topologies. This paper proposes CAMONET, a Moth-Flame Optimization based clustering "
        "scheme that elects stable cluster heads by modeling candidate selection as moths spiraling "
        "toward a flame. Simulation results show improved cluster stability and reduced control "
        "overhead compared to classical probabilistic clustering."
    ),
    "J34 (Harris Hawks)": (
        "This paper presents a Harris Hawks Optimization based clustering algorithm for vehicular "
        "ad-hoc networks. Inspired by the cooperative surprise-pounce hunting strategy of Harris hawks, "
        "the proposed method converges on cluster-head candidates from multiple directions "
        "simultaneously. Results on standard VANET mobility traces show superior cluster-head stability "
        "and lower re-election frequency compared to Grey Wolf and Moth-Flame based approaches."
    ),
    "J16 (UAV-VANET)": (
        "Urban vehicular networks suffer coverage gaps due to building obstructions and roadside unit "
        "density limits. This paper proposes a UAV-assisted communication architecture in which an "
        "aerial relay node dynamically repositions toward regions of high vehicle density, extending "
        "effective network coverage. Results show improved packet delivery ratio and reduced end-to-end "
        "delay compared to a roadside-unit-only baseline."
    ),
}
for k, v in abstracts.items():
    print(f"[{k}]\n{v}\n")

In [ ]:
synthesis_prompt = f'''Given these three paper abstracts, identify ONE open research gap that connects
all three. For every claim you make, state exactly which abstract supports it and quote the exact
supporting phrase in double quotes.

{chr(10).join(f"[{k}] {v}" for k, v in abstracts.items())}
'''
print(synthesis_prompt)

try:
    deepseek_synthesis = ask_deepseek(synthesis_prompt)
    print("\n--- DeepSeek's response ---\n")
    print(deepseek_synthesis)
except NameError:
    deepseek_synthesis = "PASTE_RESPONSE_HERE"
    print("(ask_deepseek not available — paste the prompt above into chat.deepseek.com,")
    print(" then set deepseek_synthesis = '''...''' in the next cell with the pasted response)")

In [ ]:
# If you used chat.deepseek.com instead of the API, paste the response here and uncomment:
# deepseek_synthesis = '''PASTE DEEPSEEK'S FULL RESPONSE HERE, KEEPING ITS QUOTATION MARKS'''

### Automated hallucination check — run this on DeepSeek's actual response

In [ ]:
def check_quotes_against_sources(llm_response, source_texts):
    combined = re.sub(r"\s+", " ", " ".join(source_texts).lower())
    quotes = re.findall(r'"([^"]{8,})"', llm_response)
    results = []
    for q in quotes:
        norm_q = re.sub(r"\s+", " ", q.strip().lower())
        results.append((q, norm_q in combined))
    return results

_results = check_quotes_against_sources(deepseek_synthesis, list(abstracts.values()))

if not _results:
    print("No quoted phrases (in \"double quotes\") were found in the response.")
    print("Try re-running the prompt, or explicitly ask DeepSeek to quote its sources.")
else:
    for q, found in _results:
        status = "VERIFIED ✅" if found else "⚠️  NOT FOUND IN SOURCE — possible hallucination"
        print(f"[{status}] \"{q}\"")
    n_verified = sum(f for _, f in _results)
    print(f"\n{n_verified}/{len(_results)} quoted claims verified against the actual abstracts.")
    if n_verified < len(_results):
        print("Any flagged quote should NOT be cited in real work without manually re-checking it —")
        print("this is expected and is the whole point of the exercise, not a failure.")

**Discussion (no code needed):** Even a "VERIFIED" quote being present verbatim doesn't guarantee
DeepSeek used it *in context correctly* — read the surrounding sentence in the abstract yourself.
What's the difference between "this phrase appears in the source" and "this claim is actually true"?

## Exercise C — LLM-Assisted Hybrid-Algorithm Design & Benchmarking (25 min)

Use DeepSeek as a brainstorming partner, then implement and benchmark the idea yourself using your
own Grey Wolf clustering code from Notebook 2.

**Step 1.** Run the prompt below.

In [ ]:
hybrid_prompt = '''I have two VANET clustering algorithms:
1) Grey Wolf Optimization - pack hierarchy (alpha/beta/delta/omega) guides candidate solutions
   toward the best-known cluster-head positions.
2) Harris Hawks Optimization - a cooperative "surprise pounce" strategy converges on cluster heads
   from multiple directions simultaneously.

Propose ONE concrete, implementable way to hybridize these two for VANET clustering. Explain the
expected trade-off versus each algorithm alone, and outline the idea in pseudocode using no more
than 15 lines.'''
print(hybrid_prompt)

try:
    print("\n--- DeepSeek's response ---\n")
    print(ask_deepseek(hybrid_prompt))
except NameError:
    print("(ask_deepseek not available — paste the prompt above into chat.deepseek.com instead)")

**Step 2.** Copy your working `gwo_update_position`, `fitness`, and `run_gwo` from Notebook 2 into the
cell below (or `%run` the notebook if your environment supports it), then modify `run_gwo` — call it
`run_hybrid` — to implement DeepSeek's suggested change. Aim for a small, targeted edit, not a rewrite.

In [ ]:
# Paste your Notebook 2 functions here: gwo_update_position, fitness, make_nodes, run_gwo
# ... (omitted here — copy from Notebook 2) ...

def run_hybrid(nodes, k, iterations, pop_size, seed):
    """YOUR modification of run_gwo implementing DeepSeek's hybrid idea."""
    # YOUR CODE HERE — start by copying run_gwo, then apply the change DeepSeek suggested
    pass

### Benchmark your hybrid against your Notebook 2 baseline results

In [ ]:
# Re-run the same 15-snapshot comparison from Notebook 2, adding your hybrid as a third bar.
# If gwo_results and random_results are still in memory from Notebook 2, reuse them; otherwise
# re-paste that loop here first.

N_SNAPSHOTS = 15
K = 6
hybrid_results = []
try:
    for seed in range(N_SNAPSHOTS):
        nodes = make_nodes(60, seed=200 + seed)
        _, history = run_hybrid(nodes, K, iterations=40, pop_size=15, seed=seed)
        hybrid_results.append(history[-1])

    print(f"GWO average (from Notebook 2):    {np.mean(gwo_results):.1f}")
    print(f"Hybrid average:                   {np.mean(hybrid_results):.1f}")
    improvement = 100 * (np.mean(gwo_results) - np.mean(hybrid_results)) / np.mean(gwo_results)
    print(f"\nHybrid vs. plain GWO: {improvement:+.1f}% (positive = hybrid is better)")
except NameError as e:
    print("Missing a function or variable from Notebook 2 — make sure you've pasted",
          "gwo_update_position, fitness, make_nodes, and gwo_results into this notebook.", e)

### There is no "must pass" assertion here — and that's intentional

A result showing your hybrid **did not** beat plain GWO is a completely valid, reportable finding —
record it and your hypothesis for why, exactly as you would in a real research log. Bring your number
(better or worse) and your hypothesis to the afternoon showcase.

---
## Responsible LLM Use in Research — keep these in mind going forward

- Always verify factual claims, especially citations, dates, and numbers, against the primary source
- Disclose AI assistance according to your institution's and your target journal's policy
- Never paste unpublished results or confidential data into a public LLM chat interface
- Use LLMs for first drafts, brainstorming, and debugging — not as the final word on correctness
- Keep a record of prompts for anything that ends up influencing a publication

---
**You're done with the core exercises.** Bring your Notebook 1 chart, Notebook 2 bar chart, and this
notebook's hybrid result (whatever it shows) to the project showcase.